<a href="https://colab.research.google.com/github/AKSHAYAVISWA/HexaTraining/blob/main/June%2013/%20Capstone/Telecom_Customer_Usage_and_Billing_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [54]:
%%writefile customers.csv

customer_id,customer_name,city,state,age,gender,plan_id,status
101,Rahul Sharma,Hyderabad,Telangana,35,Male,P101,Active
102,Priya Reddy,Bangalore,Karnataka,29,Female,P102,Active
103,Amit Kumar,Mumbai,Maharashtra,42,Male,P103,Inactive
104,Sneha Patel,Chennai,Tamil Nadu,31,Female,P101,Active
105,Farhan Ali,Delhi,Delhi,55,Male,P104,Active
106,Neha Singh,Pune,Maharashtra,38,Female,P102,Active
107,Arjun Verma,Hyderabad,Telangana,26,Male,P103,Inactive
108,Meera Nair,Kochi,Kerala,48,Female,P104,Active
109,Kiran Rao,Bangalore,Karnataka,33,Male,P101,Active
110,Nisha Reddy,Delhi,Delhi,41,Female,P102,Active
111,Ravi Kumar,Mumbai,Maharashtra,45,Male,P105,Active
112,Ayesha Khan,Hyderabad,Telangana,28,Female,,Active

Overwriting customers.csv


In [55]:
%%writefile usage.csv

usage_id,customer_id,usage_month,data_used_gb,call_minutes,sms_count
1001,101,2026-01,45,900,120
1002,102,2026-01,30,600,80
1003,103,2026-01,12,250,40
1004,104,2026-01,55,1100,150
1005,105,2026-01,75,1500,200
1006,106,2026-01,28,500,60
1007,107,2026-01,10,200,20
1008,108,2026-01,80,1600,250
1009,109,2026-01,48,950,100
1010,110,2026-01,32,700,90
1011,120,2026-01,60,1300,140
1012,101,2026-02,50,1000,130
1013,102,2026-02,34,650,85
1014,104,2026-02,58,1200,160
1015,105,2026-02,,1450,210

Overwriting usage.csv


In [56]:
%%writefile plans.json

[
 {
  "plan_id":"P101",
  "plan_name":"Smart Basic",
  "monthly_fee":499,
  "data_limit_gb":50,
  "features":{
   "unlimited_calls":true,
   "ott_included":false,
   "roaming":"National"
  }
 },
 {
  "plan_id":"P102",
  "plan_name":"Smart Plus",
  "monthly_fee":799,
  "data_limit_gb":75,
  "features":{
   "unlimited_calls":true,
   "ott_included":true,
   "roaming":"National"
  }
 },
 {
  "plan_id":"P103",
  "plan_name":"Budget Saver",
  "monthly_fee":299,
  "data_limit_gb":25,
  "features":{
   "unlimited_calls":false,
   "ott_included":false,
   "roaming":null
  }
 },
 {
  "plan_id":"P104",
  "plan_name":"Premium Max",
  "monthly_fee":1199,
  "data_limit_gb":100,
  "features":{
   "unlimited_calls":true,
   "ott_included":true,
   "roaming":"International"
  }
 }
]

Overwriting plans.json


In [57]:
%%writefile payments.csv

payment_id,customer_id,bill_month,amount_paid,payment_mode,payment_status
5001,101,2026-01,499,UPI,Success
5002,102,2026-01,799,Card,Success
5003,103,2026-01,299,Cash,Failed
5004,104,2026-01,499,UPI,Success
5005,105,2026-01,1199,Card,Success
5006,106,2026-01,799,UPI,Success
5007,107,2026-01,299,Cash,Pending
5008,108,2026-01,1199,Card,Success
5009,109,2026-01,499,UPI,Success
5010,110,2026-01,799,UPI,Success
5011,112,2026-01,,UPI,Success
5012,101,2026-02,499,Card,Success
5013,102,2026-02,799,UPI,Success
5014,104,2026-02,499,UPI,Success
5015,105,2026-02,1199,,Pending

Overwriting payments.csv


In [58]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Telecom Analytics") \
    .getOrCreate()

In [59]:
customers_df = spark.read.csv(
    "customers.csv",
    header=True,
    inferSchema=True
)

customers_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|
+-----------+-------------+---------+-----------+---+------+-------+--------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|  Active|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|  Active|
|        110|  Nisha Reddy|    Delhi|      Delhi| 41|Female|   P

In [60]:
usage_df = spark.read.csv(
    "usage.csv",
    header=True,
    inferSchema=True
)

usage_df.show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1001|        101|2026-01-01 00:00:00|          45|         900|      120|
|    1002|        102|2026-01-01 00:00:00|          30|         600|       80|
|    1003|        103|2026-01-01 00:00:00|          12|         250|       40|
|    1004|        104|2026-01-01 00:00:00|          55|        1100|      150|
|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|
|    1006|        106|2026-01-01 00:00:00|          28|         500|       60|
|    1007|        107|2026-01-01 00:00:00|          10|         200|       20|
|    1008|        108|2026-01-01 00:00:00|          80|        1600|      250|
|    1009|        109|2026-01-01 00:00:00|          48|         950|      100|
|    1010|        110|2026-01-01 00:00:00|          

In [61]:
payments_df = spark.read.csv(
    "payments.csv",
    header=True,
    inferSchema=True
)

payments_df.show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|
|      5008|        108|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5009|        109|2026-01-01 00:00:00|        499|         

In [62]:
plans_df = spark.read.option(
    "multiline", "true"
).json("plans.json")

plans_df.show(truncate=False)

+-------------+---------------------------+-----------+-------+------------+
|data_limit_gb|features                   |monthly_fee|plan_id|plan_name   |
+-------------+---------------------------+-----------+-------+------------+
|50           |{false, National, true}    |499        |P101   |Smart Basic |
|75           |{true, National, true}     |799        |P102   |Smart Plus  |
|25           |{false, NULL, false}       |299        |P103   |Budget Saver|
|100          |{true, International, true}|1199       |P104   |Premium Max |
+-------------+---------------------------+-----------+-------+------------+



In [63]:
print("CUSTOMERS")
customers_df.printSchema()

print("USAGE")
usage_df.printSchema()

print("PAYMENTS")
payments_df.printSchema()

print("PLANS")
plans_df.printSchema()

CUSTOMERS
root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- status: string (nullable = true)

USAGE
root
 |-- usage_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- usage_month: timestamp (nullable = true)
 |-- data_used_gb: integer (nullable = true)
 |-- call_minutes: integer (nullable = true)
 |-- sms_count: integer (nullable = true)

PAYMENTS
root
 |-- payment_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- bill_month: timestamp (nullable = true)
 |-- amount_paid: integer (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- payment_status: string (nullable = true)

PLANS
root
 |-- data_limit_gb: long (nullable = true)
 |-- features: struct (nullable = true)
 |    |-- ott_include

In [64]:
print("Customers Count:", customers_df.count())
print("Usage Count:", usage_df.count())
print("Payments Count:", payments_df.count())
print("Plans Count:", plans_df.count())

Customers Count: 12
Usage Count: 15
Payments Count: 15
Plans Count: 4


In [65]:
customers_df.write.mode("overwrite").parquet("bronze/customers")

usage_df.write.mode("overwrite").parquet("bronze/usage")

payments_df.write.mode("overwrite").parquet("bronze/payments")

plans_df.write.mode("overwrite").parquet("bronze/plans")

In [66]:
customers_df.filter(
    customers_df.plan_id.isNull()
).show()

+-----------+-------------+---------+---------+---+------+-------+------+
|customer_id|customer_name|     city|    state|age|gender|plan_id|status|
+-----------+-------------+---------+---------+---+------+-------+------+
|        112|  Ayesha Khan|Hyderabad|Telangana| 28|Female|   NULL|Active|
+-----------+-------------+---------+---------+---+------+-------+------+



In [67]:
usage_df.filter(
    usage_df.data_used_gb.isNull()
).show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1015|        105|2026-02-01 00:00:00|        NULL|        1450|      210|
+--------+-----------+-------------------+------------+------------+---------+



In [68]:
payments_df.filter(
    payments_df.amount_paid.isNull()
).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5011|        112|2026-01-01 00:00:00|       NULL|         UPI|       Success|
+----------+-----------+-------------------+-----------+------------+--------------+



In [69]:
payments_df.filter(
    payments_df.payment_mode.isNull()
).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5015|        105|2026-02-01 00:00:00|       1199|        NULL|       Pending|
+----------+-----------+-------------------+-----------+------------+--------------+



In [70]:
usage_df = usage_df.fillna({
    "data_used_gb": 0
})

usage_df.show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1001|        101|2026-01-01 00:00:00|          45|         900|      120|
|    1002|        102|2026-01-01 00:00:00|          30|         600|       80|
|    1003|        103|2026-01-01 00:00:00|          12|         250|       40|
|    1004|        104|2026-01-01 00:00:00|          55|        1100|      150|
|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|
|    1006|        106|2026-01-01 00:00:00|          28|         500|       60|
|    1007|        107|2026-01-01 00:00:00|          10|         200|       20|
|    1008|        108|2026-01-01 00:00:00|          80|        1600|      250|
|    1009|        109|2026-01-01 00:00:00|          48|         950|      100|
|    1010|        110|2026-01-01 00:00:00|          

In [71]:
payments_df = payments_df.fillna({
    "amount_paid": 0
})

payments_df.show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|
|      5008|        108|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5009|        109|2026-01-01 00:00:00|        499|         

In [72]:
payments_df = payments_df.fillna({
    "payment_mode": "Not Provided"
})

payments_df.show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|
|      5008|        108|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5009|        109|2026-01-01 00:00:00|        499|         

In [73]:
customers_df = customers_df.fillna({
    "plan_id": "UNKNOWN"
})

customers_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|
+-----------+-------------+---------+-----------+---+------+-------+--------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|  Active|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|  Active|
|        110|  Nisha Reddy|    Delhi|      Delhi| 41|Female|   P

In [74]:
from pyspark.sql.functions import when, col

customers_df = customers_df.withColumn(
    "data_quality_status",
    when(col("plan_id") == "UNKNOWN", "Incomplete")
    .otherwise("Valid")
)

customers_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|              Valid|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|              Valid|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|              Valid|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|              Valid|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|              Valid|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|              Valid|
|        108|   Meer

In [75]:
usage_df = usage_df.withColumn(
    "data_quality_status",
    when(col("data_used_gb") == 0, "Incomplete")
    .otherwise("Valid")
)

usage_df.show()

+--------+-----------+-------------------+------------+------------+---------+-------------------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+--------+-----------+-------------------+------------+------------+---------+-------------------+
|    1001|        101|2026-01-01 00:00:00|          45|         900|      120|              Valid|
|    1002|        102|2026-01-01 00:00:00|          30|         600|       80|              Valid|
|    1003|        103|2026-01-01 00:00:00|          12|         250|       40|              Valid|
|    1004|        104|2026-01-01 00:00:00|          55|        1100|      150|              Valid|
|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|              Valid|
|    1006|        106|2026-01-01 00:00:00|          28|         500|       60|              Valid|
|    1007|        107|2026-01-01 00:00:00|          10|         200|       20|              Valid|
|    1008|

In [76]:
payments_df = payments_df.withColumn(
    "data_quality_status",
    when(
        (col("amount_paid") == 0) |
        (col("payment_mode") == "Not Provided"),
        "Incomplete"
    )
    .otherwise("Valid")
)

payments_df.show()

+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|              Valid|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|              Valid|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|              Valid|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|              Valid|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|              Valid|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|              Valid|
|      5007|        107|2026-01-01 00:00:00|        299

In [77]:
customers_df.write \
    .mode("overwrite") \
    .parquet("silver/customers")

usage_df.write \
    .mode("overwrite") \
    .parquet("silver/usage")

payments_df.write \
    .mode("overwrite") \
    .parquet("silver/payments")

In [78]:
spark.read.parquet("silver/customers").show()

spark.read.parquet("silver/usage").show()

spark.read.parquet("silver/payments").show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|              Valid|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|              Valid|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|              Valid|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|              Valid|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|              Valid|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|              Valid|
|        108|   Meer

In [79]:
from pyspark.sql.functions import col, sum

customers_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in customers_df.columns
]).show()

usage_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in usage_df.columns
]).show()

payments_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in payments_df.columns
]).show()

+-----------+-------------+----+-----+---+------+-------+------+-------------------+
|customer_id|customer_name|city|state|age|gender|plan_id|status|data_quality_status|
+-----------+-------------+----+-----+---+------+-------+------+-------------------+
|          0|            0|   0|    0|  0|     0|      0|     0|                  0|
+-----------+-------------+----+-----+---+------+-------+------+-------------------+

+--------+-----------+-----------+------------+------------+---------+-------------------+
|usage_id|customer_id|usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+--------+-----------+-----------+------------+------------+---------+-------------------+
|       0|          0|          0|           0|           0|        0|                  0|
+--------+-----------+-----------+------------+------------+---------+-------------------+

+----------+-----------+----------+-----------+------------+--------------+-------------------+
|payment_id|customer_i

In [80]:
from pyspark.sql.functions import col

plans_flat_df = plans_df.select(
    "plan_id",
    "plan_name",
    "monthly_fee",
    "data_limit_gb",
    col("features.unlimited_calls").alias("unlimited_calls"),
    col("features.ott_included").alias("ott_included"),
    col("features.roaming").alias("roaming")
)

plans_flat_df.show(truncate=False)

+-------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |
+-------+------------+-----------+-------------+---------------+------------+-------------+
|P101   |Smart Basic |499        |50           |true           |false       |National     |
|P102   |Smart Plus  |799        |75           |true           |true        |National     |
|P103   |Budget Saver|299        |25           |false          |false       |NULL         |
|P104   |Premium Max |1199       |100          |true           |true        |International|
+-------+------------+-----------+-------------+---------------+------------+-------------+



In [81]:
plans_flat_df.select(
    "plan_id",
    "unlimited_calls"
).show()

+-------+---------------+
|plan_id|unlimited_calls|
+-------+---------------+
|   P101|           true|
|   P102|           true|
|   P103|          false|
|   P104|           true|
+-------+---------------+



In [82]:
plans_flat_df.select(
    "plan_id",
    "ott_included"
).show()

+-------+------------+
|plan_id|ott_included|
+-------+------------+
|   P101|       false|
|   P102|        true|
|   P103|       false|
|   P104|        true|
+-------+------------+



In [83]:
plans_flat_df.select(
    "plan_id",
    "roaming"
).show()

+-------+-------------+
|plan_id|      roaming|
+-------+-------------+
|   P101|     National|
|   P102|     National|
|   P103|         NULL|
|   P104|International|
+-------+-------------+



In [84]:
plans_flat_df = plans_flat_df.fillna({
    "roaming": "Not Available"
})

plans_flat_df.show(truncate=False)

+-------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |
+-------+------------+-----------+-------------+---------------+------------+-------------+
|P101   |Smart Basic |499        |50           |true           |false       |National     |
|P102   |Smart Plus  |799        |75           |true           |true        |National     |
|P103   |Budget Saver|299        |25           |false          |false       |Not Available|
|P104   |Premium Max |1199       |100          |true           |true        |International|
+-------+------------+-----------+-------------+---------------+------------+-------------+



In [85]:
plans_flat_df.write \
    .mode("overwrite") \
    .parquet("silver/plans_flat")

In [86]:
spark.read.parquet(
    "silver/plans_flat"
).show(truncate=False)

+-------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |
+-------+------------+-----------+-------------+---------------+------------+-------------+
|P101   |Smart Basic |499        |50           |true           |false       |National     |
|P102   |Smart Plus  |799        |75           |true           |true        |National     |
|P103   |Budget Saver|299        |25           |false          |false       |Not Available|
|P104   |Premium Max |1199       |100          |true           |true        |International|
+-------+------------+-----------+-------------+---------------+------------+-------------+



In [87]:
customers_df = spark.read.parquet("silver/customers")
usage_df = spark.read.parquet("silver/usage")
payments_df = spark.read.parquet("silver/payments")
plans_flat_df = spark.read.parquet("silver/plans_flat")

In [88]:
customer_plan_df = customers_df.join(
    plans_flat_df,
    "plan_id",
    "left"
)

customer_plan_df.show()

+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|customer_id|customer_name|     city|      state|age|gender|  status|data_quality_status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|
+-------+-----------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+
|   P101|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|  Active|              Valid| Smart Basic|        499|           50|           true|       false|     National|
|   P102|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|  Active|              Valid|  Smart Plus|        799|           75|           true|        true|     National|
|   P103|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|Inactive|              Valid|Bud

In [89]:
customer_usage_df = customers_df.join(
    usage_df,
    "customer_id",
    "left"
)

customer_usage_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+--------+-------------------+------------+------------+---------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|    1012|2026-02-01 00:00:00|          50|        1000|      130|              Valid|
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|    1001|2026-01-01 00:00:00|          45|         900|      120|              Valid|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|        

In [90]:
customer_payment_df = customers_df.join(
    payments_df,
    "customer_id",
    "left"
)

customer_payment_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+--------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|      5012|2026-02-01 00:00:00|        499|        Card|       Success|              Valid|
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|              Valid|      5001|2026-01-01 00:00:00|        499|         UPI|       Success|              Valid|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Fe

In [91]:
complete_df = customers_df \
    .join(plans_flat_df, "plan_id", "left") \
    .join(usage_df, "customer_id", "left") \
    .join(payments_df, "customer_id", "left")

complete_df.show(truncate=False)

+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+-------------------+
|customer_id|plan_id|customer_name|city     |state      |age|gender|status  |data_quality_status|plan_name   |monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming      |usage_id|usage_month        |data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|bill_month         |amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+-------+-------------+---------+-----------+---+------+--------+-------------------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-------------------+----------+-------------

In [92]:
invalid_plan_df = customers_df.join(
    plans_flat_df,
    "plan_id",
    "left_anti"
)

invalid_plan_df.show()

+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+
|plan_id|customer_id|customer_name|     city|      state|age|gender|status|data_quality_status|
+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+
|   P105|        111|   Ravi Kumar|   Mumbai|Maharashtra| 45|  Male|Active|              Valid|
|UNKNOWN|        112|  Ayesha Khan|Hyderabad|  Telangana| 28|Female|Active|         Incomplete|
+-------+-----------+-------------+---------+-----------+---+------+------+-------------------+



In [93]:
invalid_usage_df = usage_df.join(
    customers_df,
    "customer_id",
    "left_anti"
)

invalid_usage_df.show()

+-----------+--------+-------------------+------------+------------+---------+-------------------+
|customer_id|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|data_quality_status|
+-----------+--------+-------------------+------------+------------+---------+-------------------+
|        120|    1011|2026-01-01 00:00:00|          60|        1300|      140|              Valid|
+-----------+--------+-------------------+------------+------------+---------+-------------------+



In [94]:
invalid_payment_df = payments_df.join(
    customers_df,
    "customer_id",
    "left_anti"
)

invalid_payment_df.show()

+-----------+----------+----------+-----------+------------+--------------+-------------------+
|customer_id|payment_id|bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+----------+----------+-----------+------------+--------------+-------------------+
+-----------+----------+----------+-----------+------------+--------------+-------------------+



In [95]:
invalid_payment_df.show()

+-----------+----------+----------+-----------+------------+--------------+-------------------+
|customer_id|payment_id|bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+----------+----------+-----------+------------+--------------+-------------------+
+-----------+----------+----------+-----------+------------+--------------+-------------------+



In [96]:
invalid_payment_df.show()

+-----------+----------+----------+-----------+------------+--------------+-------------------+
|customer_id|payment_id|bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+----------+----------+-----------+------------+--------------+-------------------+
+-----------+----------+----------+-----------+------------+--------------+-------------------+



In [109]:
from pyspark.sql.functions import *

In [110]:
complete_df = complete_df.withColumn(
    "usage_category",
    when(col("data_used_gb") >= 70, "Heavy User")
    .when(col("data_used_gb") >= 30, "Medium User")
    .otherwise("Low User")
)

In [111]:
complete_df = complete_df.withColumn(
    "payment_category",
    when(col("amount_paid") >= 1000, "High Payment")
    .when(col("amount_paid") >= 500, "Medium Payment")
    .otherwise("Low Payment")
)

In [112]:
complete_df = complete_df.withColumn(
    "churn_risk",
    when(
        (col("status") == "Inactive") |
        (col("payment_status") != "Success"),
        "High Risk"
    )
    .when(
        col("data_used_gb") < 15,
        "Medium Risk"
    )
    .otherwise("Low Risk")
)

In [113]:
complete_df = complete_df.withColumn(
    "over_usage_gb",
    col("data_used_gb") - col("data_limit_gb")
)

In [114]:
complete_df = complete_df.withColumn(
    "over_usage_flag",
    when(col("over_usage_gb") > 0, "Yes")
    .otherwise("No")
)

In [115]:
complete_df.select(
    "customer_id",
    "usage_category",
    "payment_category",
    "churn_risk",
    "over_usage_gb",
    "over_usage_flag"
).show()

+-----------+--------------+----------------+-----------+-------------+---------------+
|customer_id|usage_category|payment_category| churn_risk|over_usage_gb|over_usage_flag|
+-----------+--------------+----------------+-----------+-------------+---------------+
|        101|   Medium User|     Low Payment|   Low Risk|            0|             No|
|        101|   Medium User|     Low Payment|   Low Risk|            0|             No|
|        101|   Medium User|     Low Payment|   Low Risk|           -5|             No|
|        101|   Medium User|     Low Payment|   Low Risk|           -5|             No|
|        102|   Medium User|  Medium Payment|   Low Risk|          -41|             No|
|        102|   Medium User|  Medium Payment|   Low Risk|          -41|             No|
|        102|   Medium User|  Medium Payment|   Low Risk|          -45|             No|
|        102|   Medium User|  Medium Payment|   Low Risk|          -45|             No|
|        103|      Low User|    

In [116]:
complete_df.write \
    .mode("overwrite") \
    .parquet("gold/telecom_transformed")

In [122]:
from pyspark.sql.functions import (
    count,
    countDistinct,
    sum,
    avg,
    round,
    desc,
    col
)

In [123]:
complete_df = spark.read.parquet("gold/telecom_transformed")

In [124]:
complete_df.groupBy("city") \
    .agg(countDistinct("customer_id").alias("total_customers")) \
    .show()

+---------+---------------+
|     city|total_customers|
+---------+---------------+
|Bangalore|              2|
|    Kochi|              1|
|  Chennai|              1|
|   Mumbai|              2|
|     Pune|              1|
|    Delhi|              2|
|Hyderabad|              3|
+---------+---------------+



In [125]:
complete_df.groupBy("state") \
    .agg(countDistinct("customer_id").alias("total_customers")) \
    .show()

+-----------+---------------+
|      state|total_customers|
+-----------+---------------+
|  Karnataka|              2|
|     Kerala|              1|
| Tamil Nadu|              1|
|      Delhi|              2|
|Maharashtra|              3|
|  Telangana|              3|
+-----------+---------------+



In [126]:
complete_df.groupBy("plan_name") \
    .agg(countDistinct("customer_id").alias("total_customers")) \
    .show()

+------------+---------------+
|   plan_name|total_customers|
+------------+---------------+
|        NULL|              2|
| Smart Basic|              3|
|Budget Saver|              2|
| Premium Max|              2|
|  Smart Plus|              3|
+------------+---------------+



In [127]:
complete_df.groupBy("usage_category") \
    .agg(countDistinct("customer_id").alias("total_customers")) \
    .show()

+--------------+---------------+
|usage_category|total_customers|
+--------------+---------------+
|   Medium User|              5|
|    Heavy User|              2|
|      Low User|              6|
+--------------+---------------+



In [128]:
complete_df.groupBy("churn_risk") \
    .agg(countDistinct("customer_id").alias("total_customers")) \
    .show()

+-----------+---------------+
| churn_risk|total_customers|
+-----------+---------------+
|   Low Risk|             10|
|Medium Risk|              1|
|  High Risk|              3|
+-----------+---------------+



In [129]:
complete_df.groupBy("plan_name") \
    .agg(sum("data_used_gb").alias("total_data_usage")) \
    .show()

+------------+----------------+
|   plan_name|total_data_usage|
+------------+----------------+
|        NULL|            NULL|
| Smart Basic|             464|
|Budget Saver|              22|
| Premium Max|             230|
|  Smart Plus|             188|
+------------+----------------+



In [130]:
complete_df.groupBy("plan_name") \
    .agg(round(avg("data_used_gb"),2).alias("average_data_usage")) \
    .show()

+------------+------------------+
|   plan_name|average_data_usage|
+------------+------------------+
|        NULL|              NULL|
| Smart Basic|             51.56|
|Budget Saver|              11.0|
| Premium Max|              46.0|
|  Smart Plus|             31.33|
+------------+------------------+



In [131]:
complete_df.groupBy("city") \
    .agg(sum("call_minutes").alias("total_call_minutes")) \
    .show()

+---------+------------------+
|     city|total_call_minutes|
+---------+------------------+
|Bangalore|              3450|
|    Kochi|              1600|
|  Chennai|              4600|
|   Mumbai|               250|
|     Pune|               500|
|    Delhi|              6600|
|Hyderabad|              4000|
+---------+------------------+



In [132]:
complete_df.groupBy("state") \
    .agg(sum("sms_count").alias("total_sms_count")) \
    .show()

+-----------+---------------+
|      state|total_sms_count|
+-----------+---------------+
|  Karnataka|            430|
|     Kerala|            250|
| Tamil Nadu|            620|
|      Delhi|            910|
|  Telangana|            520|
|Maharashtra|            100|
+-----------+---------------+



In [133]:
complete_df.filter(
    col("payment_status") == "Success"
).agg(
    sum("amount_paid").alias("total_successful_revenue")
).show()

+------------------------+
|total_successful_revenue|
+------------------------+
|                   12882|
+------------------------+



In [134]:
complete_df.groupBy("city") \
    .agg(sum("amount_paid").alias("total_revenue")) \
    .orderBy(desc("total_revenue")) \
    .show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|    Delhi|         5595|
|Bangalore|         3695|
|Hyderabad|         2295|
|  Chennai|         1996|
|    Kochi|         1199|
|     Pune|          799|
|   Mumbai|          299|
+---------+-------------+



In [135]:
complete_df.groupBy("plan_name") \
    .agg(sum("amount_paid").alias("total_revenue")) \
    .orderBy(desc("total_revenue")) \
    .show()

+------------+-------------+
|   plan_name|total_revenue|
+------------+-------------+
| Premium Max|         5995|
|  Smart Plus|         4794|
| Smart Basic|         4491|
|Budget Saver|          598|
|        NULL|            0|
+------------+-------------+



In [136]:
complete_df.groupBy("payment_mode") \
    .agg(sum("amount_paid").alias("total_revenue")) \
    .orderBy(desc("total_revenue")) \
    .show()

+------------+-------------+
|payment_mode|total_revenue|
+------------+-------------+
|         UPI|         6689|
|        Card|         6193|
|Not Provided|         2398|
|        Cash|          598|
|        NULL|         NULL|
+------------+-------------+



In [137]:
complete_df.groupBy("plan_name") \
    .agg(sum("amount_paid").alias("total_revenue")) \
    .orderBy(desc("total_revenue")) \
    .limit(1) \
    .show()

+-----------+-------------+
|  plan_name|total_revenue|
+-----------+-------------+
|Premium Max|         5995|
+-----------+-------------+



In [138]:
complete_df.groupBy("city") \
    .agg(sum("amount_paid").alias("total_revenue")) \
    .orderBy(desc("total_revenue")) \
    .limit(1) \
    .show()

+-----+-------------+
| city|total_revenue|
+-----+-------------+
|Delhi|         5595|
+-----+-------------+



In [140]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

In [141]:
usage_window = Window.orderBy(desc("data_used_gb"))

rank_usage_df = complete_df.withColumn(
    "usage_rank",
    rank().over(usage_window)
)

rank_usage_df.select(
    "customer_id",
    "customer_name",
    "data_used_gb",
    "usage_rank"
).show()

+-----------+-------------+------------+----------+
|customer_id|customer_name|data_used_gb|usage_rank|
+-----------+-------------+------------+----------+
|        108|   Meera Nair|          80|         1|
|        105|   Farhan Ali|          75|         2|
|        105|   Farhan Ali|          75|         2|
|        104|  Sneha Patel|          58|         4|
|        104|  Sneha Patel|          58|         4|
|        104|  Sneha Patel|          55|         6|
|        104|  Sneha Patel|          55|         6|
|        101| Rahul Sharma|          50|         8|
|        101| Rahul Sharma|          50|         8|
|        109|    Kiran Rao|          48|        10|
|        101| Rahul Sharma|          45|        11|
|        101| Rahul Sharma|          45|        11|
|        102|  Priya Reddy|          34|        13|
|        102|  Priya Reddy|          34|        13|
|        110|  Nisha Reddy|          32|        15|
|        102|  Priya Reddy|          30|        16|
|        102

In [142]:
payment_window = Window.orderBy(desc("amount_paid"))

rank_payment_df = complete_df.withColumn(
    "payment_rank",
    rank().over(payment_window)
)

rank_payment_df.select(
    "customer_id",
    "customer_name",
    "amount_paid",
    "payment_rank"
).show()

+-----------+-------------+-----------+------------+
|customer_id|customer_name|amount_paid|payment_rank|
+-----------+-------------+-----------+------------+
|        105|   Farhan Ali|       1199|           1|
|        105|   Farhan Ali|       1199|           1|
|        105|   Farhan Ali|       1199|           1|
|        105|   Farhan Ali|       1199|           1|
|        108|   Meera Nair|       1199|           1|
|        102|  Priya Reddy|        799|           6|
|        102|  Priya Reddy|        799|           6|
|        102|  Priya Reddy|        799|           6|
|        102|  Priya Reddy|        799|           6|
|        106|   Neha Singh|        799|           6|
|        110|  Nisha Reddy|        799|           6|
|        101| Rahul Sharma|        499|          12|
|        101| Rahul Sharma|        499|          12|
|        101| Rahul Sharma|        499|          12|
|        101| Rahul Sharma|        499|          12|
|        104|  Sneha Patel|        499|       

In [143]:
top3_data_users = complete_df.withColumn(
    "rank",
    rank().over(Window.orderBy(desc("data_used_gb")))
).filter(col("rank") <= 3)

top3_data_users.select(
    "customer_id",
    "customer_name",
    "data_used_gb"
).show()

+-----------+-------------+------------+
|customer_id|customer_name|data_used_gb|
+-----------+-------------+------------+
|        108|   Meera Nair|          80|
|        105|   Farhan Ali|          75|
|        105|   Farhan Ali|          75|
+-----------+-------------+------------+



In [144]:
top3_revenue = complete_df.withColumn(
    "rank",
    rank().over(Window.orderBy(desc("amount_paid")))
).filter(col("rank") <= 3)

top3_revenue.select(
    "customer_id",
    "customer_name",
    "amount_paid"
).show()

+-----------+-------------+-----------+
|customer_id|customer_name|amount_paid|
+-----------+-------------+-----------+
|        105|   Farhan Ali|       1199|
|        105|   Farhan Ali|       1199|
|        105|   Farhan Ali|       1199|
|        105|   Farhan Ali|       1199|
|        108|   Meera Nair|       1199|
+-----------+-------------+-----------+



In [145]:
city_window = Window.partitionBy("city") \
    .orderBy(desc("data_used_gb"))

top_city_customer = complete_df.withColumn(
    "rank",
    row_number().over(city_window)
).filter(col("rank") == 1)

top_city_customer.select(
    "city",
    "customer_id",
    "customer_name",
    "data_used_gb"
).show()

+---------+-----------+-------------+------------+
|     city|customer_id|customer_name|data_used_gb|
+---------+-----------+-------------+------------+
|Bangalore|        109|    Kiran Rao|          48|
|  Chennai|        104|  Sneha Patel|          58|
|    Delhi|        105|   Farhan Ali|          75|
|Hyderabad|        101| Rahul Sharma|          50|
|    Kochi|        108|   Meera Nair|          80|
|   Mumbai|        103|   Amit Kumar|          12|
|     Pune|        106|   Neha Singh|          28|
+---------+-----------+-------------+------------+



In [146]:
plan_window = Window.partitionBy("plan_name") \
    .orderBy(desc("data_used_gb"))

top_plan_customer = complete_df.withColumn(
    "rank",
    row_number().over(plan_window)
).filter(col("rank") == 1)

top_plan_customer.select(
    "plan_name",
    "customer_id",
    "customer_name",
    "data_used_gb"
).show()

+------------+-----------+-------------+------------+
|   plan_name|customer_id|customer_name|data_used_gb|
+------------+-----------+-------------+------------+
|        NULL|        111|   Ravi Kumar|        NULL|
|Budget Saver|        103|   Amit Kumar|          12|
| Premium Max|        108|   Meera Nair|          80|
| Smart Basic|        104|  Sneha Patel|          58|
|  Smart Plus|        102|  Priya Reddy|          34|
+------------+-----------+-------------+------------+



In [147]:
monthly_revenue = complete_df.groupBy(
    "bill_month"
).agg(
    sum("amount_paid").alias("monthly_revenue")
)

revenue_window = Window.orderBy("bill_month")

monthly_revenue.withColumn(
    "running_total_revenue",
    sum("monthly_revenue").over(revenue_window)
).show()

+-------------------+---------------+---------------------+
|         bill_month|monthly_revenue|running_total_revenue|
+-------------------+---------------+---------------------+
|               NULL|           NULL|                 NULL|
|2026-01-01 00:00:00|           9886|                 9886|
|2026-02-01 00:00:00|           5992|                15878|
+-------------------+---------------+---------------------+



In [148]:
lag_window = Window.partitionBy(
    "customer_id"
).orderBy("usage_month")

lag_df = complete_df.withColumn(
    "previous_month_usage",
    lag("data_used_gb", 1).over(lag_window)
)

lag_df.select(
    "customer_id",
    "usage_month",
    "data_used_gb",
    "previous_month_usage"
).show()

+-----------+-------------------+------------+--------------------+
|customer_id|        usage_month|data_used_gb|previous_month_usage|
+-----------+-------------------+------------+--------------------+
|        101|2026-01-01 00:00:00|          45|                NULL|
|        101|2026-01-01 00:00:00|          45|                  45|
|        101|2026-02-01 00:00:00|          50|                  45|
|        101|2026-02-01 00:00:00|          50|                  50|
|        102|2026-01-01 00:00:00|          30|                NULL|
|        102|2026-01-01 00:00:00|          30|                  30|
|        102|2026-02-01 00:00:00|          34|                  30|
|        102|2026-02-01 00:00:00|          34|                  34|
|        103|2026-01-01 00:00:00|          12|                NULL|
|        104|2026-01-01 00:00:00|          55|                NULL|
|        104|2026-01-01 00:00:00|          55|                  55|
|        104|2026-02-01 00:00:00|          58|  

In [149]:
lead_df = complete_df.withColumn(
    "next_month_usage",
    lead("data_used_gb", 1).over(lag_window)
)

lead_df.select(
    "customer_id",
    "usage_month",
    "data_used_gb",
    "next_month_usage"
).show()

+-----------+-------------------+------------+----------------+
|customer_id|        usage_month|data_used_gb|next_month_usage|
+-----------+-------------------+------------+----------------+
|        101|2026-01-01 00:00:00|          45|              45|
|        101|2026-01-01 00:00:00|          45|              50|
|        101|2026-02-01 00:00:00|          50|              50|
|        101|2026-02-01 00:00:00|          50|            NULL|
|        102|2026-01-01 00:00:00|          30|              30|
|        102|2026-01-01 00:00:00|          30|              34|
|        102|2026-02-01 00:00:00|          34|              34|
|        102|2026-02-01 00:00:00|          34|            NULL|
|        103|2026-01-01 00:00:00|          12|            NULL|
|        104|2026-01-01 00:00:00|          55|              55|
|        104|2026-01-01 00:00:00|          55|              58|
|        104|2026-02-01 00:00:00|          58|              58|
|        104|2026-02-01 00:00:00|       

In [150]:
increase_df = complete_df.withColumn(
    "previous_month_usage",
    lag("data_used_gb").over(lag_window)
)

increase_df.filter(
    col("data_used_gb") > col("previous_month_usage")
).select(
    "customer_id",
    "customer_name",
    "usage_month",
    "previous_month_usage",
    "data_used_gb"
).show()

+-----------+-------------+-------------------+--------------------+------------+
|customer_id|customer_name|        usage_month|previous_month_usage|data_used_gb|
+-----------+-------------+-------------------+--------------------+------------+
|        101| Rahul Sharma|2026-02-01 00:00:00|                  45|          50|
|        102|  Priya Reddy|2026-02-01 00:00:00|                  30|          34|
|        104|  Sneha Patel|2026-02-01 00:00:00|                  55|          58|
+-----------+-------------+-------------------+--------------------+------------+



In [151]:
increase_df.write \
    .mode("overwrite") \
    .parquet("gold/window_analysis")

In [152]:
customers_df.createOrReplaceTempView("customers")

usage_df.createOrReplaceTempView("usage")

payments_df.createOrReplaceTempView("payments")

plans_flat_df.createOrReplaceTempView("plans")

complete_df.createOrReplaceTempView("telecom")

In [153]:
spark.sql("""
SELECT *
FROM customers
WHERE status = 'Active'
""").show()

+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|Active|              Valid|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|Active|              Valid|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|Active|              Valid|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|Active|              Valid|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|Active|              Valid|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|Active|              Valid|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|Active|              Valid|
|        110|  Nisha Reddy|    Delhi|   

In [154]:
spark.sql("""
SELECT city,
COUNT(DISTINCT customer_id) AS total_customers
FROM customers
GROUP BY city
ORDER BY total_customers DESC
""").show()

+---------+---------------+
|     city|total_customers|
+---------+---------------+
|Hyderabad|              3|
|Bangalore|              2|
|   Mumbai|              2|
|    Delhi|              2|
|    Kochi|              1|
|  Chennai|              1|
|     Pune|              1|
+---------+---------------+



In [155]:
spark.sql("""
SELECT plan_name,
SUM(amount_paid) AS total_revenue
FROM telecom
GROUP BY plan_name
ORDER BY total_revenue DESC
""").show()

+------------+-------------+
|   plan_name|total_revenue|
+------------+-------------+
| Premium Max|         5995|
|  Smart Plus|         4794|
| Smart Basic|         4491|
|Budget Saver|          598|
|        NULL|            0|
+------------+-------------+



In [156]:
spark.sql("""
SELECT customer_id,
customer_name,
data_used_gb,
usage_category
FROM telecom
WHERE usage_category = 'Heavy User'
""").show()

+-----------+-------------+------------+--------------+
|customer_id|customer_name|data_used_gb|usage_category|
+-----------+-------------+------------+--------------+
|        105|   Farhan Ali|          75|    Heavy User|
|        105|   Farhan Ali|          75|    Heavy User|
|        108|   Meera Nair|          80|    Heavy User|
+-----------+-------------+------------+--------------+



In [157]:
spark.sql("""
SELECT customer_id,
customer_name,
status,
payment_status,
churn_risk
FROM telecom
WHERE churn_risk = 'High Risk'
""").show()

+-----------+-------------+--------+--------------+----------+
|customer_id|customer_name|  status|payment_status|churn_risk|
+-----------+-------------+--------+--------------+----------+
|        103|   Amit Kumar|Inactive|        Failed| High Risk|
|        105|   Farhan Ali|  Active|       Pending| High Risk|
|        105|   Farhan Ali|  Active|       Pending| High Risk|
|        107|  Arjun Verma|Inactive|       Pending| High Risk|
+-----------+-------------+--------+--------------+----------+



In [158]:
spark.sql("""
SELECT *
FROM customers
WHERE plan_id='P105'
OR plan_id='UNKNOWN'
""").show()

+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|status|data_quality_status|
+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+
|        111|   Ravi Kumar|   Mumbai|Maharashtra| 45|  Male|   P105|Active|              Valid|
|        112|  Ayesha Khan|Hyderabad|  Telangana| 28|Female|UNKNOWN|Active|         Incomplete|
+-----------+-------------+---------+-----------+---+------+-------+------+-------------------+



In [159]:
spark.sql("""
SELECT *
FROM payments
WHERE payment_status IN ('Failed','Pending')
""").show()

+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|              Valid|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|              Valid|
|      5015|        105|2026-02-01 00:00:00|       1199|Not Provided|       Pending|         Incomplete|
+----------+-----------+-------------------+-----------+------------+--------------+-------------------+



In [160]:
spark.sql("""
SELECT customer_id,
customer_name,
MAX(data_used_gb) AS total_usage
FROM telecom
GROUP BY customer_id, customer_name
ORDER BY total_usage DESC
LIMIT 5
""").show()

+-----------+-------------+-----------+
|customer_id|customer_name|total_usage|
+-----------+-------------+-----------+
|        108|   Meera Nair|         80|
|        105|   Farhan Ali|         75|
|        104|  Sneha Patel|         58|
|        101| Rahul Sharma|         50|
|        109|    Kiran Rao|         48|
+-----------+-------------+-----------+



In [161]:
spark.sql("""
SELECT payment_mode,
SUM(amount_paid) AS total_revenue
FROM telecom
GROUP BY payment_mode
ORDER BY total_revenue DESC
""").show()

+------------+-------------+
|payment_mode|total_revenue|
+------------+-------------+
|         UPI|         6689|
|        Card|         6193|
|Not Provided|         2398|
|        Cash|          598|
|        NULL|         NULL|
+------------+-------------+



In [162]:
spark.catalog.listTables()

[Table(name='customers', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='payments', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='plans', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='telecom', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True),
 Table(name='usage', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [163]:
complete_df.write \
    .mode("overwrite") \
    .parquet("gold/final_gold")

In [164]:
complete_df.write \
    .mode("overwrite") \
    .partitionBy("usage_month") \
    .parquet("gold/final_gold_partitioned")

In [166]:
%%writefile usage_incremental.csv

usage_id,customer_id,usage_month,data_used_gb,call_minutes,sms_count
1016,101,2026-03,55,1050,140
1017,102,2026-03,38,700,90
1018,104,2026-03,62,1250,170
1019,105,2026-03,85,1550,220
1020,108,2026-03,95,1700,280

Writing usage_incremental.csv


In [167]:
incremental_df = spark.read.csv(
    "usage_incremental.csv",
    header=True,
    inferSchema=True
)

incremental_df.show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1016|        101|2026-03-01 00:00:00|          55|        1050|      140|
|    1017|        102|2026-03-01 00:00:00|          38|         700|       90|
|    1018|        104|2026-03-01 00:00:00|          62|        1250|      170|
|    1019|        105|2026-03-01 00:00:00|          85|        1550|      220|
|    1020|        108|2026-03-01 00:00:00|          95|        1700|      280|
+--------+-----------+-------------------+------------+------------+---------+



In [168]:
silver_usage = spark.read.parquet("silver/usage")

before_count = silver_usage.count()

print("Before Count:", before_count)

Before Count: 15


In [169]:
incremental_df.write \
    .mode("append") \
    .parquet("silver/usage")

In [170]:
updated_usage = spark.read.parquet("silver/usage")

updated_usage.count()

20

In [171]:
usage_df = spark.read.parquet("silver/usage")

In [172]:
customers_clean = customers_df.drop("data_quality_status")

payments_clean = payments_df.drop("data_quality_status")

In [173]:
complete_df = customers_clean \
    .join(plans_flat_df, "plan_id", "left") \
    .join(usage_df, "customer_id", "left") \
    .join(payments_clean, "customer_id", "left")

In [174]:
from pyspark.sql.functions import *

complete_df = complete_df.withColumn(
    "usage_category",
    when(col("data_used_gb") >= 70, "Heavy User")
    .when(col("data_used_gb") >= 30, "Medium User")
    .otherwise("Low User")
)

complete_df = complete_df.withColumn(
    "payment_category",
    when(col("amount_paid") >= 1000, "High Payment")
    .when(col("amount_paid") >= 500, "Medium Payment")
    .otherwise("Low Payment")
)

complete_df = complete_df.withColumn(
    "churn_risk",
    when(
        (col("status") == "Inactive") |
        (col("payment_status") != "Success"),
        "High Risk"
    )
    .when(col("data_used_gb") < 15, "Medium Risk")
    .otherwise("Low Risk")
)

complete_df = complete_df.withColumn(
    "over_usage_gb",
    col("data_used_gb") - col("data_limit_gb")
)

complete_df = complete_df.withColumn(
    "over_usage_flag",
    when(col("over_usage_gb") > 0, "Yes")
    .otherwise("No")
)

In [175]:
complete_df.write \
    .mode("overwrite") \
    .partitionBy("usage_month") \
    .parquet("gold/final_gold_partitioned")

In [176]:
spark.read.parquet(
    "gold/final_gold_partitioned"
).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+------------+------------+---------+-------------------+----------+-------------------+-----------+------------+--------------+--------------+----------------+----------+-------------+---------------+-------------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|data_used_gb|call_minutes|sms_count|data_quality_status|payment_id|         bill_month|amount_paid|payment_mode|payment_status|usage_category|payment_category|churn_risk|over_usage_gb|over_usage_flag|        usage_month|
+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+------------+------------+---------+------------------

In [177]:
after_count = spark.read.parquet(
    "silver/usage"
).count()

print("Before Incremental Load :", before_count)

print("After Incremental Load  :", after_count)

print("New Records Added       :", after_count - before_count)

Before Incremental Load : 15
After Incremental Load  : 20
New Records Added       : 5


In [178]:
after_count = spark.read.parquet(
    "silver/usage"
).count()

print("Before Incremental Load :", before_count)

print("After Incremental Load  :", after_count)

print("New Records Added       :", after_count - before_count)

Before Incremental Load : 15
After Incremental Load  : 20
New Records Added       : 5


In [180]:
customer_usage_summary = complete_df.select(
    "customer_id",
    "customer_name",
    "city",
    "plan_name",
    "usage_month",
    "data_used_gb",
    "data_limit_gb",
    "over_usage_flag",
    "amount_paid",
    "payment_status",
    "churn_risk"
)

customer_usage_summary.show()

+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+----------+
|customer_id|customer_name|     city|   plan_name|        usage_month|data_used_gb|data_limit_gb|over_usage_flag|amount_paid|payment_status|churn_risk|
+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+----------+
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-03-01 00:00:00|          55|           50|            Yes|        499|       Success|  Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-03-01 00:00:00|          55|           50|            Yes|        499|       Success|  Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-02-01 00:00:00|          50|           50|             No|        499|       Success|  Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-02-01 00:00:00|          50|     

In [181]:
customer_usage_summary.write \
    .mode("overwrite") \
    .parquet("gold/customer_usage_summary")

In [182]:
city_revenue_report = complete_df.groupBy(
    "city"
).agg(
    countDistinct("customer_id").alias("total_customers"),
    sum("amount_paid").alias("total_revenue"),
    round(avg("amount_paid"), 2).alias("average_payment")
)

city_revenue_report.show(truncate=False)

+---------+---------------+-------------+---------------+
|city     |total_customers|total_revenue|average_payment|
+---------+---------------+-------------+---------------+
|Bangalore|2              |5293         |756.14         |
|Kochi    |1              |2398         |1199.0         |
|Chennai  |1              |2994         |499.0          |
|Mumbai   |2              |299          |299.0          |
|Pune     |1              |799          |799.0          |
|Delhi    |2              |7993         |1141.86        |
|Hyderabad|3              |3293         |411.63         |
+---------+---------------+-------------+---------------+



In [183]:
city_revenue_report.write \
    .mode("overwrite") \
    .parquet("gold/city_revenue_report")

In [184]:
churn_risk_report = complete_df.select(
    "customer_id",
    "customer_name",
    "city",
    "plan_name",
    "payment_status",
    "status",
    "churn_risk"
)

churn_risk_report.show(truncate=False)

+-----------+-------------+---------+------------+--------------+--------+----------+
|customer_id|customer_name|city     |plan_name   |payment_status|status  |churn_risk|
+-----------+-------------+---------+------------+--------------+--------+----------+
|101        |Rahul Sharma |Hyderabad|Smart Basic |Success       |Active  |Low Risk  |
|101        |Rahul Sharma |Hyderabad|Smart Basic |Success       |Active  |Low Risk  |
|101        |Rahul Sharma |Hyderabad|Smart Basic |Success       |Active  |Low Risk  |
|101        |Rahul Sharma |Hyderabad|Smart Basic |Success       |Active  |Low Risk  |
|101        |Rahul Sharma |Hyderabad|Smart Basic |Success       |Active  |Low Risk  |
|101        |Rahul Sharma |Hyderabad|Smart Basic |Success       |Active  |Low Risk  |
|102        |Priya Reddy  |Bangalore|Smart Plus  |Success       |Active  |Low Risk  |
|102        |Priya Reddy  |Bangalore|Smart Plus  |Success       |Active  |Low Risk  |
|102        |Priya Reddy  |Bangalore|Smart Plus  |Succ

In [185]:
churn_risk_report.write \
    .mode("overwrite") \
    .parquet("gold/churn_risk_report")

In [186]:
over_usage_report = complete_df.select(
    "customer_id",
    "customer_name",
    "plan_name",
    "data_used_gb",
    "data_limit_gb",
    "over_usage_gb"
).filter(
    col("over_usage_gb") > 0
)

over_usage_report.show(truncate=False)

+-----------+-------------+-----------+------------+-------------+-------------+
|customer_id|customer_name|plan_name  |data_used_gb|data_limit_gb|over_usage_gb|
+-----------+-------------+-----------+------------+-------------+-------------+
|104        |Sneha Patel  |Smart Basic|62          |50           |12           |
|104        |Sneha Patel  |Smart Basic|62          |50           |12           |
|104        |Sneha Patel  |Smart Basic|58          |50           |8            |
|104        |Sneha Patel  |Smart Basic|58          |50           |8            |
|104        |Sneha Patel  |Smart Basic|55          |50           |5            |
|104        |Sneha Patel  |Smart Basic|55          |50           |5            |
|101        |Rahul Sharma |Smart Basic|55          |50           |5            |
|101        |Rahul Sharma |Smart Basic|55          |50           |5            |
+-----------+-------------+-----------+------------+-------------+-------------+



In [189]:
from pyspark.sql.functions import *

plan_performance_report = complete_df.groupBy(
    "plan_name"
).agg(
    countDistinct("customer_id").alias("total_customers"),
    sum("data_used_gb").alias("total_data_usage"),
    round(avg("data_used_gb"), 2).alias("average_data_usage"),
    sum("amount_paid").alias("total_revenue")
)

plan_performance_report.show(truncate=False)

+------------+---------------+----------------+------------------+-------------+
|plan_name   |total_customers|total_data_usage|average_data_usage|total_revenue|
+------------+---------------+----------------+------------------+-------------+
|NULL        |2              |NULL            |NULL              |0            |
|Smart Basic |3              |698             |53.69             |6487         |
|Budget Saver|2              |22              |11.0              |598          |
|Premium Max |2              |495             |61.88             |9592         |
|Smart Plus  |3              |264             |33.0              |6392         |
+------------+---------------+----------------+------------------+-------------+



In [190]:
plan_performance_report.write \
    .mode("overwrite") \
    .parquet("gold/plan_performance_report")

In [191]:
print("Customer Usage Summary")
customer_usage_summary.show(5)

print("Plan Performance Report")
plan_performance_report.show()

print("City Revenue Report")
city_revenue_report.show()

print("Churn Risk Report")
churn_risk_report.show()

print("Over Usage Report")
over_usage_report.show()

Customer Usage Summary
+-----------+-------------+---------+-----------+-------------------+------------+-------------+---------------+-----------+--------------+----------+
|customer_id|customer_name|     city|  plan_name|        usage_month|data_used_gb|data_limit_gb|over_usage_flag|amount_paid|payment_status|churn_risk|
+-----------+-------------+---------+-----------+-------------------+------------+-------------+---------------+-----------+--------------+----------+
|        101| Rahul Sharma|Hyderabad|Smart Basic|2026-03-01 00:00:00|          55|           50|            Yes|        499|       Success|  Low Risk|
|        101| Rahul Sharma|Hyderabad|Smart Basic|2026-03-01 00:00:00|          55|           50|            Yes|        499|       Success|  Low Risk|
|        101| Rahul Sharma|Hyderabad|Smart Basic|2026-02-01 00:00:00|          50|           50|             No|        499|       Success|  Low Risk|
|        101| Rahul Sharma|Hyderabad|Smart Basic|2026-02-01 00:00:00|  

In [192]:
city_revenue_report = complete_df.groupBy(
    "city"
).agg(
    countDistinct("customer_id").alias("total_customers"),
    sum("amount_paid").alias("total_revenue"),
    round(avg("amount_paid"), 2).alias("average_payment")
)

city_revenue_report.show()

+---------+---------------+-------------+---------------+
|     city|total_customers|total_revenue|average_payment|
+---------+---------------+-------------+---------------+
|Bangalore|              2|         5293|         756.14|
|    Kochi|              1|         2398|         1199.0|
|  Chennai|              1|         2994|          499.0|
|   Mumbai|              2|          299|          299.0|
|     Pune|              1|          799|          799.0|
|    Delhi|              2|         7993|        1141.86|
|Hyderabad|              3|         3293|         411.63|
+---------+---------------+-------------+---------------+



In [193]:
churn_risk_report = complete_df.select(
    "customer_id",
    "customer_name",
    "city",
    "plan_name",
    "payment_status",
    "status",
    "churn_risk"
)

churn_risk_report.show()

+-----------+-------------+---------+------------+--------------+--------+----------+
|customer_id|customer_name|     city|   plan_name|payment_status|  status|churn_risk|
+-----------+-------------+---------+------------+--------------+--------+----------+
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|  Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|  Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|  Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|  Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|  Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|  Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|  Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|  Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|    